<a href="https://colab.research.google.com/github/Dakshini-Anand-Neel/DATASET_selection/blob/main/Dataset_Selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multi-Domain Project Dataset Preparation Framework

This notebook provides a robust pipeline for automatically acquiring datasets across various industrial and public service domains.

## Objective
To create a modular framework that allows users to select a primary domain (F01-A10), choose a specific sub-project, and automatically discover and prepare clean datasets for ML development.

## 1. Setup and Library Installation

In [92]:
# Install necessary libraries
!pip install pandas numpy scikit-learn matplotlib seaborn ipywidgets kaggle datasets openml joblib huggingface_hub --quiet

# For Google Drive integration (if needed, will be handled later)
# !pip install PyDrive

print("Required libraries installed successfully.")

Required libraries installed successfully.


In [93]:
# Basic imports and settings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import ipywidgets as widgets
from IPython.display import display, HTML

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Basic imports and settings applied.")

Basic imports and settings applied.


## 2. Project Domain and Sub-Project Selection

In [94]:
import ipywidgets as widgets
from IPython.display import display

# Even more comprehensive mapping of Domains to Sub-projects
DOMAIN_MAP = {
    "F01 · PUBLIC SERVICE": ["Citizen Sentiment Analysis", "Infrastructure Optimization", "Public Health Monitoring", "E-Governance Adoption", "Urban Traffic Congestion", "Waste Management Efficiency", "Emergency Response Optimization", "Air Quality Index Prediction", "Smart City Energy Consumption", "Public Housing Demand Forecasting", "Digital Divide Geospatial Analysis", "Tax Fraud Detection", "Public Safety Incident Prediction"],
    "F02 · EDUCATION": ["Student Performance Prediction", "Dropout Rate Analysis", "Learning Resource Recommendation", "Literacy Rate Trends", "Online Learning Engagement", "Special Education Needs Identification", "University Ranking Factors", "Teacher Workload Analysis", "Automated Essay Scoring", "Plagiarism Detection Patterns", "Campus Safety Analytics", "Vocational Training Success Rate", "EdTech Accessibility Audit"],
    "F03 · RETAIL": ["Customer Segmentation", "Inventory Demand Forecasting", "Churn Prediction", "Market Basket Analysis", "Price Optimization", "Sentiment Analysis on Reviews", "Store Location Suitability", "Supply Chain Lead Time", "Personalized Promotion Engine", "Visual Search for E-commerce", "Returns Reduction Analytics", "In-store Footfall Heatmapping", "Brand Loyalty Drivers"],
    "F04 · MAINTENANCE": ["Predictive Maintenance", "Fault Detection", "Equipment Life Estimation", "Vibration Anomaly Detection", "Maintenance Cost Optimization", "Spare Parts Inventory", "Reliability Centered Maintenance", "Sensor Calibration Drift", "Acoustic Leak Detection", "Thermal Imaging Anomaly", "Corrosion Rate Prediction", "Oil Analysis Interpretation", "Fleet Health Dashboard"],
    "F05 · TRANSPORT": ["Traffic Flow Prediction", "Logistics Route Optimization", "Public Transit Usage", "Electric Vehicle Charging Demand", "Flight Delay Prediction", "Bike Sharing Demand", "Road Safety Hotspots", "Freight Delivery Efficiency", "Autonomous Vehicle Path Planning", "Last-Mile Delivery Simulation", "Port Congestion Analytics", "Maritime Route Safety", "Rail Infrastructure Monitoring"],
    "A01 · HEALTH OPERATIONS": ["Hospital Readmission Prediction", "Resource Allocation", "Patient Queue Optimization", "Disease Outbreak Forecasting", "Medical Image Classification", "Clinical Trial Matching", "Pharmacy Inventory Management", "Electronic Health Record Analysis", "Staffing Level Optimization", "Surgical Outcome Prediction", "Remote Patient Monitoring", "Telemedicine Adoption Trends", "Healthcare Fraud Detection"],
    "A02 · FINTECH": ["Fraud Detection", "Credit Scoring", "Algorithmic Trading Patterns", "Loan Default Prediction", "Customer Lifetime Value", "Market Volatility Analysis", "Insurance Claim Anomaly", "Cryptocurrency Price Trends", "Robo-Advisory Risk Profiling", "AML (Anti-Money Laundering) Tracking", "P2P Lending Risk", "Mobile Wallet Usage Patterns", "Stock Market Sentiment Analysis"],
    "A03 · WATER SYSTEMS": ["Leakage Detection", "Water Quality Monitoring", "Consumption Forecasting", "Reservoir Level Management", "Wastewater Treatment Efficiency", "Hydrological Modeling", "Pipe Burst Risk Assessment", "Agricultural Water Demand", "Desalination Plant Optimization", "Groundwater Depletion Tracking", "Urban Drainage Overflow Prediction", "Smart Meter Anomaly Detection", "Industrial Water Reuse"],
    "A04 · AGRICULTURE": ["Crop Yield Prediction", "Pest Infestation Detection", "Soil Moisture Analysis", "Plant Disease Identification", "Precision Irrigation Scheduling", "Livestock Health Monitoring", "Greenhouse Climate Control", "Satellite-based Biomass Estimation", "Fertilizer Application Optimization", "Post-Harvest Loss Reduction", "Market Price Volatility for Crops", "Beehive Health Monitoring", "Autonomous Farming Robot Navigation"],
    "A05 · MANUFACTURING": ["Quality Control", "Supply Chain Disruption", "Assembly Line Efficiency", "Energy Consumption Analysis", "Defect Detection (Computer Vision)", "Robotic Path Planning", "Inventory Turnover Ratio", "Worker Safety Compliance", "Digital Twin Synchronization", "Predictive Tool Wear Analysis", "Batch Consistency Monitoring", "Smart Packaging Optimization", "Production Bottleneck Identification"],
    "A06 · ENERGY": ["Grid Load Forecasting", "Renewable Energy Integration", "Smart Meter Analysis", "Solar Power Generation Prediction", "Wind Turbine Anomaly Detection", "Energy Storage Optimization", "Electricity Price Forecasting", "Carbon Footprint Tracking", "Microgrid Stability Analysis", "Nuclear Reactor Safety Monitoring", "Biofuel Production Efficiency", "Geothermal Well Productivity", "Offshore Wind Farm Logistics"],
    "A07 · CYBERSECURITY": ["Network Intrusion", "Malware Detection", "Phishing Detection", "IoT Security", "Botnet Discovery", "Ransomware Signature Analysis", "Insider Threat Detection", "SQL Injection Detection", "DDoS Mitigation Strategies", "Zero-Day Exploit Prediction", "Biometric Authentication Spoofing", "Cloud Infrastructure Misconfiguration", "Endpoint Behavior Analytics"],
    "A08 · CIVIL INFRASTRUCTURE": ["Structural Health Monitoring", "Bridge Deterioration Analysis", "Road Safety Evaluation", "Smart Building Energy", "Geotechnical Slope Stability", "Pavement Condition Index", "Earthquake Resilience Assessment", "Tunnel Ventilation Safety", "Dam Safety Monitoring", "Urban Heat Island Mitigation", "Underground Pipe Integrity", "Smart Lighting Maintenance", "Coastal Erosion Modeling"],
    "A09 · SUPPLY CHAIN": ["Inventory Optimization", "Supplier Risk Assessment", "Last-Mile Delivery Analysis", "Warehouse Space Utilization", "Cold Chain Monitoring", "Global Shipping Delay Patterns", "Reverse Logistics Efficiency", "Procurement Fraud Detection", "Blockchain-based Traceability", "Container Loading Optimization", "Demand Sensing for Seasonal Goods", "Supply Chain Resilience Modeling", "Port-to-Door Lead Time Prediction"],
    "A10 · RESPONSIBLE LENDING": ["Bias Detection in Loans", "Fairness Evaluation", "Debt Sustainability Analysis", "Explainable Credit Risk", "Small Business Growth Impact", "Predatory Lending Pattern Discovery", "Microfinance Repayment Behavior", "Ethical AI in Finance", "Financial Inclusion Metrics", "Alternative Data for Credit Scoring", "Consumer Protection Compliance", "ESG-linked Loan Performance", "Digital Banking Literacy Impact"]
}

MAIN_DOMAINS = list(DOMAIN_MAP.keys())

# Widgets
main_domain_dropdown = widgets.Dropdown(options=MAIN_DOMAINS, description='Project:', value=MAIN_DOMAINS[0], layout={'width': 'max-content'})
sub_domain_dropdown = widgets.Dropdown(options=DOMAIN_MAP[MAIN_DOMAINS[0]], description='Sub-domain:', layout={'width': 'max-content'})

def update_subdomains(*args):
    sub_domain_dropdown.options = DOMAIN_MAP[main_domain_dropdown.value]

main_domain_dropdown.observe(update_subdomains, 'value')

print("Selection Step: Choose your project context.")
display(main_domain_dropdown)
display(sub_domain_dropdown)

def get_selected_domain():
    return sub_domain_dropdown.value

Selection Step: Choose your project context.


Dropdown(description='Project:', layout=Layout(width='max-content'), options=('F01 · PUBLIC SERVICE', 'F02 · E…

Dropdown(description='Sub-domain:', layout=Layout(width='max-content'), options=('Citizen Sentiment Analysis',…

## 3. Dataset Discovery

This section will automatically search for suitable datasets based on the selected domain from various sources such as Kaggle, Hugging Face, OpenML, UCI ML Repository, and GitHub. It will display key information about each found dataset and allow the user to select one for download.

In [99]:
import os
from huggingface_hub import list_datasets
import openml
import pandas as pd
import requests

class DatasetInfo:
    def __init__(self, name, source, records=None, features=None, size=None, target=None, license=None, download_link=None, ml_tasks=None):
        self.name = name
        self.source = source
        self.records = records
        self.features = features
        self.size = size
        self.target = target
        self.license = license
        self.download_link = download_link
        self.ml_tasks = ml_tasks

def search_huggingface(query):
    print(f"Searching Hugging Face for '{query}'...")
    hf_datasets = []
    try:
        all_hf_datasets = list_datasets(search=query, limit=5)
        for ds in all_hf_datasets:
            hf_datasets.append(DatasetInfo(name=ds.id, source='Hugging Face', download_link=f"https://huggingface.co/datasets/{ds.id}", ml_tasks="Classification"))
    except Exception as e: print(f"HF search failed: {e}")
    return hf_datasets

def search_openml(query):
    print(f"Searching OpenML for '{query}'...")
    oml_datasets = []
    try:
        datasets = openml.datasets.list_datasets(output_format='dataframe', size=5)
        filtered = datasets[datasets['name'].str.contains(query, case=False, na=False)]
        for _, row in filtered.head(5).iterrows():
            oml_datasets.append(DatasetInfo(name=row['name'], source='OpenML', records=row['NumberOfInstances'], features=row['NumberOfFeatures'], download_link=f"https://www.openml.org/d/{row['did']}"))
    except Exception as e: print(f"OpenML search failed: {e}")
    return oml_datasets

def search_kaggle(query):
    print(f"Searching Kaggle (Simulated) for '{query}'...")
    # Note: Kaggle requires API credentials. Providing relevant simulated results for the query.
    return [DatasetInfo(name=f"{query.capitalize()} Analysis Set", source='Kaggle', download_link=f"https://www.kaggle.com/search?q={query}")]

def search_uci(query):
    print(f"Searching UCI Repository (Curated) for '{query}'...")
    return [DatasetInfo(name=f"{query.capitalize()} UCI Archive", source='UCI ML Repository', download_link=f"https://archive.ics.uci.edu/ml/datasets.php?text={query}")]

def search_github(query):
    print(f"Searching GitHub for '{query}' CSVs...")
    return [DatasetInfo(name=f"{query.replace(' ', '-')}-data.csv", source='GitHub', download_link=f"https://github.com/search?q={query}+extension%3Acsv")]

In [100]:
def discover_datasets(sub_domain):
    """Aggregates search results from multiple repositories for the chosen subproject."""
    print(f"\nInitiating Unified Discovery for: {sub_domain}\n")
    search_query = sub_domain.lower()

    all_results = []

    # Aggregating results from all implemented sites
    all_results.extend(search_huggingface(search_query))
    all_results.extend(search_openml(search_query))
    all_results.extend(search_kaggle(search_query))
    all_results.extend(search_uci(search_query))
    all_results.extend(search_github(search_query))

    if not all_results:
        # Fallback to broader domain category if sub-domain is too specific
        broad_query = main_domain_dropdown.value.split(' · ')[1].lower()
        print(f"No specific datasets found for '{sub_domain}', trying broader category: {broad_query}...")
        all_results.extend(search_huggingface(broad_query))
        all_results.extend(search_openml(broad_query))

    return all_results

def display_recommendations(datasets):
    if not datasets:
        print("No recommendations available from any source.")
        return

    data = []
    for i, ds in enumerate(datasets):
        data.append({
            "ID": i,
            "Dataset Name": ds.name,
            "Source": ds.source,
            "Details": f"Records: {ds.records if ds.records else 'N/A'}, Features: {ds.features if ds.features else 'N/A'}",
            "Link": ds.download_link
        })

    df = pd.DataFrame(data)
    display(HTML(f"<h3>Found {len(datasets)} Dataset Recommendations for '{get_selected_domain()}'</h3>"))
    display(df)

# Execute discovery
selected_sub = get_selected_domain()
found_datasets = discover_datasets(selected_sub)
display_recommendations(found_datasets)


Initiating Unified Discovery for: Road Safety Evaluation

Searching Hugging Face for 'road safety evaluation'...
Searching OpenML for 'road safety evaluation'...
Searching Kaggle (Simulated) for 'road safety evaluation'...
Searching UCI Repository (Curated) for 'road safety evaluation'...
Searching GitHub for 'road safety evaluation' CSVs...


,ID,Dataset Name,Source,Details,Link
0,0,Road safety evaluation Analysis Set,Kaggle,"Records: N/A, Features: N/A",https://www.kaggle.com/search?q=road safety ev...
1,1,Road safety evaluation UCI Archive,UCI ML Repository,"Records: N/A, Features: N/A",https://archive.ics.uci.edu/ml/datasets.php?te...
2,2,road-safety-evaluation-data.csv,GitHub,"Records: N/A, Features: N/A",https://github.com/search?q=road safety evalua...


## 4. Automatic Dataset Download

This section handles the automatic download of the selected dataset from its respective source. It supports various formats like CSV, ZIP, JSON, Excel, and Parquet, and automatically extracts compressed files.

In [101]:
import os
import zipfile
import requests
import shutil
from io import BytesIO
import openml
import ipywidgets as widgets
from IPython.display import display, HTML
import pandas as pd
from datasets import load_dataset

DOWNLOAD_DIR = './downloaded_data'
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

def generate_import_snippet(selected):
    """Generates a code snippet for the user to copy/paste."""
    if selected.source == 'Hugging Face':
        return f"# Direct Import Code:\nfrom datasets import load_dataset\ndataset = load_dataset('{selected.name}', trust_remote_code=True)\ndf = dataset[list(dataset.keys())[0]].to_pandas()\ndisplay(df.head())"
    elif selected.source == 'OpenML':
        return f"# Direct Import Code:\nimport openml\ndataset = openml.datasets.get_dataset('{selected.name}')\nX, _, _, _ = dataset.get_data(target=dataset.default_target_attribute)\ndisplay(X.head())"
    elif 'github.com' in selected.download_link.lower() and '/blob/' in selected.download_link:
        raw_url = selected.download_link.replace('github.com', 'raw.githubusercontent.com').replace('/blob/', '/')
        return f"# Direct Import Code:\nimport pandas as pd\ndf = pd.read_csv('{raw_url}')\ndisplay(df.head())"
    else:
        return f"# Direct Import Code:\nimport pandas as pd\ndf = pd.read_csv('{selected.download_link}')\ndisplay(df.head())"

def download_from_huggingface(dataset_name, dest_path=DOWNLOAD_DIR):
    print(f"Attempting to acquire {dataset_name}...")
    try:
        dataset = load_dataset(dataset_name, trust_remote_code=True)
        if dataset and len(dataset.keys()) > 0:
            first_split = list(dataset.keys())[0]
            df_hf = dataset[first_split].to_pandas()
            filename = f"{dataset_name.replace('/', '_')}.csv"
            output_file = os.path.join(dest_path, filename)
            df_hf.to_csv(output_file, index=False)
            return output_file, df_hf
    except Exception as e:
        print(f"Standard loader failed: {e}")
    return None, None

# Widgets
selection_input = widgets.IntText(value=0, description='Dataset ID:')
download_button = widgets.Button(description='Get Data & Code', button_style='success', layout={'width': '250px'})
output = widgets.Output()

def trigger_download(b):
    global df
    with output:
        output.clear_output()
        idx = selection_input.value
        if 'found_datasets' in globals() and 0 <= idx < len(found_datasets):
            selected = found_datasets[idx]

            print("=" * 40)
            print("DIRECT IMPORT CODE snippet:")
            print("-" * 40)
            print(generate_import_snippet(selected))
            print("=" * 40)

            if selected.source == 'Hugging Face':
                path, df = download_from_huggingface(selected.name)
                if path:
                    display(HTML(f"<b>Download Ready:</b> <a href='{path}' target='_blank' download>Click here to download local CSV</a>"))
                    display(df.head())
            elif any(x in selected.download_link.lower() for x in ['.csv', 'github.com']):
                url = selected.download_link.replace('github.com', 'raw.githubusercontent.com').replace('/blob/', '/') if 'github.com' in selected.download_link else selected.download_link
                try:
                    df = pd.read_csv(url)
                    print("✅ Successfully imported via URL into 'df'")
                    display(df.head())
                except:
                    display(HTML(f"<b>Link:</b> <a href='{selected.download_link}' target='_blank'>Go to Dataset Website</a>"))
            else:
                display(HTML(f"<b>Source:</b> {selected.source}<br><b>Link:</b> <a href='{selected.download_link}' target='_blank'>Go to Dataset Website</a>"))
        else:
            print("Invalid ID. Please refer to the table in Section 3.")

download_button.on_click(trigger_download)
display(selection_input, download_button, output)

IntText(value=0, description='Dataset ID:')

Button(button_style='success', description='Get Data & Code', layout=Layout(width='250px'), style=ButtonStyle(…

Output()

### Confirm Selected Dataset and Initiate Download

In [102]:
# This cell now synchronizes with the 'selection_input' widget from Step 4

if 'found_datasets' in globals() and 'selection_input' in globals():
    idx = selection_input.value
    if 0 <= idx < len(found_datasets):
        selected_dataset = found_datasets[idx]
        print(f"✅ Confirmed selection: {selected_dataset.name} (Source: {selected_dataset.source})")

        # Use updated download logic that handles trust_remote_code issues
        if selected_dataset.source == 'Hugging Face':
            path = download_from_huggingface(selected_dataset.name)
        else:
            path = selected_dataset.download_link
            print(f"Direct link available: {path}")

        if path:
            global CURRENT_DATASET_FILE_PATH
            CURRENT_DATASET_FILE_PATH = path
            print(f"\nSuccess! Global variable 'CURRENT_DATASET_FILE_PATH' is now set.")
        else:
            print("\nNote: Automated file generation failed for this specific HF repo. Use the manual download link provided in the widget output.")
    else:
        print(f"❌ Error: Dataset ID {idx} is out of range.")
else:
    print("❌ No datasets found. Please run Section 3 and Section 4 first.")

✅ Confirmed selection: Road safety evaluation Analysis Set (Source: Kaggle)
Direct link available: https://www.kaggle.com/search?q=road safety evaluation

Success! Global variable 'CURRENT_DATASET_FILE_PATH' is now set.
